# ex02 · 多层感知机的从零实现（对应教材 4.2）

> **做题流程**：按部分补全 TODO，每个自测跑出 ✓ 再继续；做完再看 `solutions/ex02-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）
>
> 本节从零实现一个两层 MLP（隐藏层 256 + ReLU），在 Fashion-MNIST 上做分类。
> 与上一章 softmax 从零的**唯一区别**：中间多了一层隐藏层 + 激活函数。

In [1]:
import torch
import torchvision
from torch.utils import data
from torchvision import transforms

def load_data_fashion_mnist(batch_size):
    trans = transforms.ToTensor()
    mnist_train = torchvision.datasets.FashionMNIST(
        root='../data', train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(
        root='../data', train=False, transform=trans, download=True)
    return (data.DataLoader(mnist_train, batch_size, shuffle=True),
            data.DataLoader(mnist_test, batch_size, shuffle=False))

batch_size = 256
train_iter, test_iter = load_data_fashion_mnist(batch_size)
print('训练集批数:', len(train_iter))

训练集批数: 235


## 第一部分 · 初始化参数（TODO 2.1）

先预测：W1 的形状为什么是 (784, 256)？b1 为什么是 (256,)？

In [3]:
num_inputs, num_hiddens, num_outputs = 784, 256, 10

# TODO 2.1: 初始化 W1(784,256)、b1(256)、W2(256,10)、b2(10)
# W1/W2 用 torch.normal(0, 0.01, size=(...))，b1/b2 用 torch.zeros(...)，都要 requires_grad=True
W1 = torch.normal(0, 0.01, size=(784, 256), requires_grad = True)
W2 = torch.normal(0, 0.01, size=(256, 10), requires_grad = True)
b1 = torch.zeros((256,), requires_grad = True)
b2 = torch.zeros((10,), requires_grad = True)             

## 第二部分 · 激活函数与模型（TODO 2.2 ~ 2.3）

补全 relu 和 net。注意 net 的返回是 **logits**（未经过 softmax），softmax 交给损失函数去做。

先预测：为什么 net 里要先 reshape(-1, 784)？隐藏层做了什么事？

In [8]:
def relu(X):
    # TODO 2.2: torch.max(X, torch.zeros_like(X))
    return torch.max(X, torch.zeros_like(X))


def net(X):
    # TODO 2.3: X 展平成 (batch, 784) → 隐藏层 relu(X@W1+b1) → 输出层 H@W2+b2
    X_flattened = X.reshape(-1, 784)
    H = relu(X_flattened @ W1 + b1)
    return H @ W2 + b2 

### 自测：完成 TODO 2.2~2.3 后运行

In [9]:
try:
    assert relu(torch.tensor([[-1.0, 2.0]])).tolist() == [[0.0, 2.0]]
    out = net(torch.zeros(4, 1, 28, 28))
    assert list(out.shape) == [4, 10], f'net 输出形状不对: {out.shape}'
    print('✓ relu 与 net 正确，输出形状', list(out.shape))
except NotImplementedError as e:
    print(f'⚠ {e}')
except AssertionError as e:
    print(f'✗ {e}')

✓ relu 与 net 正确，输出形状 [4, 10]


## 第三部分 · 损失 / 准确率 / 优化器（给定，ch03 ex07 已写过）

这里直接复用。注意损失用 `nn.CrossEntropyLoss(reduction='none')`（内部自带 softmax，所以 net 不用再套 softmax）。

In [ ]:
from torch import nn

loss = nn.CrossEntropyLoss(reduction='none')

def accuracy(y_hat, y):
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())

def evaluate_accuracy(net, data_iter):
    metric = 0.0
    n = 0
    with torch.no_grad():
        for X, y in data_iter:
            metric += accuracy(net(X), y)
            n += len(X)
    return metric / n

def sgd(params, lr, batch_size):
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

## 第四部分 · 训练循环（TODO 2.4）

和 softmax 从零的循环几乎一样，只是参数从 [w, b] 变成 [W1, b1, W2, b2]。

先预测：MLP 的测试准确率会比上一章 softmax 从零（~0.85）高还是低？

In [ ]:
lr = 0.1
num_epochs = 10

try:
    for epoch in range(num_epochs):
        train_acc_sum = 0.0
        n = 0
        for X, y in train_iter:
            y_hat = net(X)
            # TODO 2.4: 计算交叉熵损失 l = loss(y_hat, y)；l.sum().backward()；sgd 更新 [W1,b1,W2,b2]
            raise NotImplementedError('⚠ TODO 2.4: 训练循环未完成')
            train_acc_sum += accuracy(y_hat, y)
            n += len(X)
        test_acc = evaluate_accuracy(net, test_iter)
        print(f'epoch {epoch + 1}, train acc {train_acc_sum / n:.3f}, test acc {test_acc:.3f}')
except NotImplementedError as e:
    print(f'⚠ {e}，先完成 TODO 2.4 再运行')

## 小结与面试衔接

- MLP = 多层全连接 + 激活函数；从零版只比 softmax 从零多一个隐藏层
- 隐藏层在做什么（面试高频）：把原始像素「变换」成更适合分类的中间表示，最后一层在这个表示上做线性决策
- 为什么 MLP 比 softmax 回归强：多了一层非线性，能拟合更复杂的决策边界